In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import zipfile
import os

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
# Download the 4 csv physical data

df_1 = pd.read_csv('../results/cleaned/Physical dataset/phy_att_1_cleaned.csv', delimiter=',', header=0, encoding='utf-8')
df_2 = pd.read_csv('../results/cleaned/Physical dataset/phy_att_2_cleaned.csv', delimiter=',', header=0, encoding='utf-8')
df_3 = pd.read_csv('../results/cleaned/Physical dataset/phy_att_3_cleaned.csv', delimiter=',', header=0, encoding='utf-8')
df_4 = pd.read_csv('../results/cleaned/Physical dataset/phy_att_4_cleaned.csv', delimiter=',', header=0, encoding='utf-8')

In [3]:
# Remove Pump_3 column and Valv_1 to Valv_9 columns, because they are all set to False

columns_to_remove = ['Time','Pump_3', 'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7', 'Valv_8', 'Valv_9']

df_1 = df_1.drop(columns=columns_to_remove)
df_2 = df_2.drop(columns=columns_to_remove)
df_3 = df_3.drop(columns=columns_to_remove)
df_4 = df_4.drop(columns=columns_to_remove)

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report, f1_score, balanced_accuracy_score, matthews_corrcoef
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import pandas as pd
import plotly.express as px
import numpy as np
import time

In [74]:
def knn_model_with_plots(df, index, k_max=20, k_for_conf_matrix=5):
    """
    Perform KNN analysis with accuracy plots and confusion matrix for a specific k.

    Parameters:
        df (pd.DataFrame): The input dataframe with features and target labels.
        index (int or str): An identifier for the dataframe (optional).
        k_max (int): Maximum number of neighbors to test.
        k_for_conf_matrix (int): Specific k for confusion matrix plot.

    Returns:
        dict: A dictionary containing metrics and confusion matrix for the specific k.
    """
    # Prepare the data
    features = df.drop(['Label', 'Label_n'], axis=1)
    target = df['Label']

    # Normalize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # SMOTE to balance the data
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(features_scaled, target)

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X_resampled, y_resampled, test_size=0.3, random_state=42
    )

    results = []  
    confusion_matrices = {}  
    total_training_time = 0

    # Loop for different values of k
    for k in range(1, k_max + 1):
        knn = KNeighborsClassifier(n_neighbors=k)

        # Measure training time
        start_time = time.time()
        knn.fit(X_train, y_train)
        training_time = time.time() - start_time
        total_training_time += training_time

        # Predictions
        y_pred = knn.predict(X_test)

        # Accuracy and confusion matrix
        accuracy = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)

        results.append({"k": k, "accuracy": accuracy, "training_time": training_time})
        confusion_matrices[k] = cm

    # Average training time
    train_time_avg = total_training_time / k_max

    # Convert results to DataFrame
    results_df = pd.DataFrame(results)

    # Plot accuracy vs k
    fig_acc = px.line(
        results_df,
        x="k",
        y="accuracy",
        title=f"Training Accuracy vs k (KNN) for DataFrame {index} | Avg Training Time: {train_time_avg:.4f}s",
        labels={"k": "Number of Neighbors (k)", "accuracy": "Accuracy"},
        markers=True,
    )
    fig_acc.update_traces(mode="lines+markers")  # Enhance graph readability
    fig_acc.show()

    # Confusion matrix for k = k_for_conf_matrix
    if k_for_conf_matrix in confusion_matrices:
        cm = confusion_matrices[k_for_conf_matrix]
        fig_cm = px.imshow(
            cm,
            text_auto="int",
            labels=dict(x="Predicted", y="True", color="Count"),
            title=f"Confusion Matrix (KNN, k={k_for_conf_matrix}) for DataFrame {index}",
            color_continuous_scale="viridis"
        )
        fig_cm.show()
    else:
        print(f"No confusion matrix found for k={k_for_conf_matrix}")

    # Save figures
    output_dir = f"../results/figures/Physical dataset/df_{index}"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    fig_acc.write_image(os.path.join(output_dir, f"knn_accuracy.png"))
    if k_for_conf_matrix in confusion_matrices:
        fig_cm.write_image(os.path.join(output_dir, f"knn_conf_matrix_k_{k_for_conf_matrix}.png"))

    print(f"Figures saved at: {output_dir}")
    print(f"Average training time: {train_time_avg:.4f} seconds")

    return {
        "Metrics": results_df,
        "Confusion Matrix (k=5)": confusion_matrices.get(k_for_conf_matrix, None)
    }


In [ ]:
# KNN on all df
results_1 = knn_model_with_plots(df_1, index=1)
results_2 = knn_model_with_plots(df_2, index=2)
results_3 = knn_model_with_plots(df_3, index=3)
results_4 = knn_model_with_plots(df_4, index=4)

In [78]:
def cart_model_analysis(df, class_weights=None, index=None, max_depth=5):
    """
    Perform CART analysis on a given dataframe with optional class weighting.

    Parameters:Physical dataset
        df (pd.DataFrame): The input dataframe with features and target labels.
        class_weights (str or dict): Class weighting. Use 'balanced' or a dictionary with class weights.
        index (int or str): An identifier for the dataframe (optional).
        max_depth (int): Maximum depth of the CART model.

    Returns:
        dict: A dictionary containing key metrics and the confusion matrix.
    """
    # Prepare data
    X = df.drop(columns=['Label_n', 'Label'])  # Features
    y = df['Label']  # Target

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Create and train CART model with class weighting
    cart_model = DecisionTreeClassifier(
        random_state=42, 
        max_depth=max_depth, 
        class_weight=class_weights
    )
    start_time = time.time()
    cart_model.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predictions
    y_pred = cart_model.predict(X_test)

    # Metrics
    metrics = {
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred, average='weighted'),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

    # Generate classification report
    report = classification_report(y_test, y_pred, output_dict=True)

    # Confusion matrix
    conf_matrix = confusion_matrix(y_test, y_pred)

    # Plot confusion matrix
    fig_cm = px.imshow(
        conf_matrix,
        text_auto="int",
        labels=dict(x="Predicted", y="True", color="Count"),
        title=f"Confusion Matrix (CART) for DataFrame {index}",
        color_continuous_scale="rdylbu"
    )
    fig_cm.show()

    # Create a metrics DataFrame for plotting
    metrics_df = pd.DataFrame.from_dict(metrics, orient='index', columns=['Score'])
    fig_metrics = px.bar(
        metrics_df,
        x=metrics_df.index,
        y="Score",
        text="Score",
        title=f"Model Metrics (CART) for DataFrame {index} / Training Time : {training_time: .4f} / {class_weights}",
        labels={"index": "Metric", "Score": "Value"},
        color="Score",
        color_continuous_scale="viridis"
    )
    fig_metrics.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    fig_metrics.show()

    # Save figures
    output_dir = f"../results/figures/Physical dataset/df_{index}"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    fig_cm.write_image(os.path.join(output_dir, f"cart_conf_matrix_{class_weights}.png"))
    fig_metrics.write_image(os.path.join(output_dir, f"cart_metrics_{class_weights}.png"))

    print(f"Figures saved at: {output_dir}")

    return {
        "Metrics": metrics,
        "Classification Report": report,
        "Confusion Matrix": conf_matrix
    }


In [ ]:
# CART on all df
cart_model_analysis(df_1, class_weights='balanced', index=1, max_depth=20)
cart_model_analysis(df_2, class_weights='balanced', index=2, max_depth=20)
cart_model_analysis(df_3, class_weights='balanced', index=3, max_depth=20)
cart_model_analysis(df_4, class_weights='balanced', index=4, max_depth=20)  

In [85]:
def random_forest_model_analysis(df, class_weights=None, index=None, n_estimators=100, max_depth=None):
    """
    Perform Random Forest analysis on a given dataframe with optional class weighting.

    Parameters:
        df (pd.DataFrame): The input dataframe with features and target labels.
        class_weights (str or dict): Class weighting. Use 'balanced', 'balanced_subsample', or a dictionary with class weights.
        index (int or str): An identifier for the dataframe (optional).
        n_estimators (int): The number of trees in the forest.
        max_depth (int or None): The maximum depth of the trees.

    Returns:
        dict: A dictionary containing key metrics and the confusion matrix.
    """
    # Prepare data
    X = df.drop(columns=['Label_n', 'Label'])  # Features
    y = df['Label']  # Target

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Create and train Random Forest model with class weighting
    rf_model = RandomForestClassifier(
        random_state=42,
        n_estimators=n_estimators,
        max_depth=max_depth,
        class_weight=class_weights
    )
    start_time = time.time()
    rf_model.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predictions
    y_pred = rf_model.predict(X_test)

    # Metrics
    metrics = {
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred, average='weighted'),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

    # Generate classification report
    report = classification_report(y_test, y_pred, output_dict=True)

    # Confusion matrix
    conf_matrix = confusion_matrix(y_test, y_pred)

    # Plot confusion matrix
    fig_cm = px.imshow(
        conf_matrix,
        text_auto="int",
        labels=dict(x="Predicted", y="True", color="Count"),
        title=f"Confusion Matrix (Random Forest) for DataFrame {index}",
        color_continuous_scale="rdylbu"
    )
    fig_cm.show()

    # Create a metrics DataFrame for plotting
    metrics_df = pd.DataFrame.from_dict(metrics, orient='index', columns=['Score'])
    fig_metrics = px.bar(
        metrics_df,
        x=metrics_df.index,
        y="Score",
        text="Score",
        title=f"Model Metrics (Random Forest) for DataFrame {index} / Training Time: {training_time:.4f}s / {class_weights}",
        labels={"index": "Metric", "Score": "Value"},
        color="Score",
        color_continuous_scale="viridis"
    )
    fig_metrics.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    fig_metrics.show()

    # Save figures
    output_dir = f"../results/figures/Physical dataset/df_{index}"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    fig_cm.write_image(os.path.join(output_dir, f"rf_conf_matrix_{class_weights}.png"))
    fig_metrics.write_image(os.path.join(output_dir, f"rf_metrics_{class_weights}.png"))

    print(f"Figures saved at: {output_dir}")

    return {
        "Metrics": metrics,
        "Classification Report": report,
        "Confusion Matrix": conf_matrix
    }


In [ ]:
# Exemple pour analyser un dataset spécifique
random_forest_model_analysis(df_1, class_weights='balanced', index=1, n_estimators=100, max_depth=20)
random_forest_model_analysis(df_2, class_weights='balanced', index=2, n_estimators=100, max_depth=20)
random_forest_model_analysis(df_3, class_weights='balanced', index=3, n_estimators=100, max_depth=20)
random_forest_model_analysis(df_4, class_weights='balanced', index=4, n_estimators=100, max_depth=20)


In [10]:
import xgboost as xgb

def xgboost_model_analysis(df, class_weights=None, index=None, n_estimators=100, max_depth=6, learning_rate=0.1, subsample=1.0):
    """
    Perform XGBoost analysis on a given dataframe with optional class weighting.

    Parameters:
        df (pd.DataFrame): The input dataframe with features and target labels.
        class_weights (dict or None): Class weighting. Provide a dictionary with class weights or None.
        index (int or str): An identifier for the dataframe (optional).
        n_estimators (int): The number of boosting rounds.
        max_depth (int): Maximum depth of the trees.
        learning_rate (float): Step size shrinkage to prevent overfitting.
        subsample (float): Subsample ratio of the training instances (0 < subsample <= 1).

    Returns:
        dict: A dictionary containing key metrics and the confusion matrix.
    """
    # Prepare data
    X = df.drop(columns=['Label_n', 'Label'])  # Features
    y = df['Label']  # Target

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)


    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Handle class weights
    if class_weights:
        weight_dict = class_weights
        sample_weights = y_train.map(weight_dict).values
    else:
        sample_weights = None

    # Create and train XGBoost model
    xgb_model = xgb.XGBClassifier(
        random_state=42,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        objective='multi:softmax',
        eval_metric='mlogloss',
        use_label_encoder=False
    )

    start_time = time.time()
    xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
    training_time = time.time() - start_time

    # Predictions
    y_pred = xgb_model.predict(X_test)

    # Metrics
    metrics = {
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred, average='weighted'),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

    # Generate classification report
    report = classification_report(y_test, y_pred, output_dict=True)

    # Confusion matrix
    conf_matrix = confusion_matrix(y_test, y_pred)

    # Plot confusion matrix
    fig_cm = px.imshow(
        conf_matrix,
        text_auto="int",
        labels=dict(x="Predicted", y="True", color="Count"),
        title=f"Confusion Matrix (XGBoost) for DataFrame {index}",
        color_continuous_scale="rdylbu"
    )
    fig_cm.show()

    # Create a metrics DataFrame for plotting
    metrics_df = pd.DataFrame.from_dict(metrics, orient='index', columns=['Score'])
    fig_metrics = px.bar(
        metrics_df,
        x=metrics_df.index,
        y="Score",
        text="Score",
        title=f"Model Metrics (XGBoost) for DataFrame {index} / Training Time: {training_time:.4f}s",
        labels={"index": "Metric", "Score": "Value"},
        color="Score",
        color_continuous_scale="viridis"
    )
    fig_metrics.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    fig_metrics.show()

    # Save figures
    output_dir = f"../results/figures/Physical dataset/df_{index}"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    fig_cm.write_image(os.path.join(output_dir, f"xgb_conf_matrix.png"))
    fig_metrics.write_image(os.path.join(output_dir, f"xgb_metrics.png"))

    print(f"Figures saved at: {output_dir}")

    return {
        "Metrics": metrics,
        "Classification Report": report,
        "Confusion Matrix": conf_matrix
    }


In [ ]:
xgboost_model_analysis(df_1, class_weights=None, index=1, n_estimators=100, max_depth=6, learning_rate=0.1, subsample=1.0)
xgboost_model_analysis(df_2, class_weights=None, index=2, n_estimators=100, max_depth=6, learning_rate=0.1, subsample=1.0)
xgboost_model_analysis(df_3, class_weights=None, index=3, n_estimators=100, max_depth=6, learning_rate=0.1, subsample=1.0)
xgboost_model_analysis(df_4, class_weights=None, index=4, n_estimators=100, max_depth=6, learning_rate=0.1, subsample=1.0)

In [22]:
def mlp_model_analysis(df, index=None, hidden_layer_sizes=(100,), max_iter=200, alpha=0.0001, learning_rate='constant'):
    """
    Perform MLP (Multi-Layer Perceptron) analysis on a given dataframe with optional class weighting.

    Parameters:
        df (pd.DataFrame): The input dataframe with features and target labels.
        class_weights (str or dict): Class weighting. Use 'balanced' or a dictionary with class weights.
        index (int or str): An identifier for the dataframe (optional).
        hidden_layer_sizes (tuple): The ith element represents the number of neurons in the ith hidden layer.
        max_iter (int): Maximum number of iterations for the optimizer.
        alpha (float): L2 regularization term (weight decay).
        learning_rate (str): Learning rate schedule ('constant', 'invscaling', 'adaptive').

    Returns:
        dict: A dictionary containing key metrics and the confusion matrix.
    """
    # Prepare data
    X = df.drop(columns=['Label_n', 'Label'])  # Features
    y = df['Label']  # Target
    
    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)  # Encodes strings to integers

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    # Create and train MLP model
    mlp_model = MLPClassifier(
        random_state=42,
        hidden_layer_sizes=hidden_layer_sizes,
        max_iter=max_iter,
        alpha=alpha,
        learning_rate=learning_rate,
    )

    start_time = time.time()
    mlp_model.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predictions
    y_pred = mlp_model.predict(X_test)

    # Decode predictions back to original labels
    y_pred_decoded = label_encoder.inverse_transform(y_pred)
    y_test_decoded = label_encoder.inverse_transform(y_test)

    # Metrics
    metrics = {
        "Balanced Accuracy": balanced_accuracy_score(y_test_decoded, y_pred_decoded),
        "F1-Score": f1_score(y_test_decoded, y_pred_decoded, average='weighted'),
        "MCC": matthews_corrcoef(y_test_decoded, y_pred_decoded)
    }

    # Generate classification report
    report = classification_report(y_test_decoded, y_pred_decoded, output_dict=True)

    # Confusion matrix
    conf_matrix = confusion_matrix(y_test_decoded, y_pred_decoded)

    # Plot confusion matrix
    fig_cm = px.imshow(
        conf_matrix,
        text_auto="int",
        labels=dict(x="Predicted", y="True", color="Count"),
        title=f"Confusion Matrix (MLP) for DataFrame {index}",
        color_continuous_scale="rdylbu"
    )
    fig_cm.show()

    # Create a metrics DataFrame for plotting
    metrics_df = pd.DataFrame.from_dict(metrics, orient='index', columns=['Score'])
    fig_metrics = px.bar(
        metrics_df,
        x=metrics_df.index,
        y="Score",
        text="Score",
        title=f"Model Metrics (MLP) for DataFrame {index} / Training Time : {training_time: .4f}",
        labels={"index": "Metric", "Score": "Value"},
        color="Score",
        color_continuous_scale="viridis"
    )
    fig_metrics.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    fig_metrics.show()

    # Save figures
    output_dir = f"../results/figures/Physical dataset/df_{index}"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    fig_cm.write_image(os.path.join(output_dir, f"mlp_conf_matrix.png"))
    fig_metrics.write_image(os.path.join(output_dir, f"mlp_metrics.png"))

    print(f"Figures saved at: {output_dir}")

    return {
        "Metrics": metrics,
        "Classification Report": report,
        "Confusion Matrix": conf_matrix
    }

In [23]:
mlp_model_analysis(df_1, index=1, hidden_layer_sizes=(100,), max_iter=200, alpha=0.0001, learning_rate='constant')
mlp_model_analysis(df_2, index=2, hidden_layer_sizes=(100,), max_iter=200, alpha=0.0001, learning_rate='constant')
mlp_model_analysis(df_3, index=3, hidden_layer_sizes=(100,), max_iter=200, alpha=0.0001, learning_rate='constant')
mlp_model_analysis(df_4, index=4, hidden_layer_sizes=(100,), max_iter=200, alpha=0.0001, learning_rate='constant')

Figures saved at: ../results/figures/Physical dataset/df_1


/home/johan/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning:

Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/home/johan/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning:

Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/home/johan/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning:

Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.



Figures saved at: ../results/figures/Physical dataset/df_2


Figures saved at: ../results/figures/Physical dataset/df_3


Figures saved at: ../results/figures/Physical dataset/df_4


{'Metrics': {'Balanced Accuracy': 0.666668318474833,
  'F1-Score': 0.9265757532881139,
  'MCC': 0.8596531316177886},
 'Classification Report': {'DoS': {'precision': 0.9655172413793104,
   'recall': 0.9032258064516129,
   'f1-score': 0.9333333333333333,
   'support': 31.0},
  'MITM': {'precision': 1.0,
   'recall': 0.9245283018867925,
   'f1-score': 0.9607843137254902,
   'support': 53.0},
  'normal': {'precision': 0.916,
   'recall': 0.9870689655172413,
   'f1-score': 0.950207468879668,
   'support': 232.0},
  'physical fault': {'precision': 1.0,
   'recall': 0.5185185185185185,
   'f1-score': 0.6829268292682926,
   'support': 27.0},
  'scan': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 1.0},
  'accuracy': 0.9302325581395349,
  'macro avg': {'precision': 0.7763034482758621,
   'recall': 0.666668318474833,
   'f1-score': 0.7054503890413567,
   'support': 344.0},
  'weighted avg': {'precision': 0.9373344025661587,
   'recall': 0.9302325581395349,
   'f1-score': 0.926575